# Lezione 8 — Pratica: QAOA e Max-Cut

Nella lezione teorica hai addestrato un circuito QAOA con una sola ripetizione del ciclo costo-mescolamento (`p = 1`) su un ciclo di sei nodi. Ora verifica tu stesso se aggiungere una seconda ripetizione (`p = 2`) migliora la qualità del taglio che ottieni.

## Traccia

Cambia `p = 1` in `p = 2` nel circuito QAOA della lezione teorica. Con due ripetizioni servono due simboli di costo e due di mescolamento, e due chiamate a `tfq.util.exponential` concatenate con `+=`, una per ogni ripetizione del ciclo. Riaddestra il modello e confronta il taglio medio ottenuto con quello di `p = 1`. Annota se la soluzione migliora e metti in relazione il risultato con l'idea che più ripetizioni avvicinino la soluzione all'ottimo.

**Suggerimenti**

- Dichiara quattro simboli con `sympy.symbols("gamma1 gamma2 beta1 beta2")`
- La prima chiamata a `tfq.util.exponential` usa `gamma1` e `beta1`; la seconda usa `gamma2` e `beta2`; concatenale con `circuito_qaoa += ...` due volte
- Il resto della pipeline (modello, `compile`, `fit`, campionamento, confronto con la forza bruta) è identico a quello di `p = 1`: cambiano solo il circuito e la lista di simboli che passi a `tfq.layers.Sample`
- Riutilizza lo stesso grafo (ciclo di sei nodi) e gli stessi `hamiltoniano_coste` / `hamiltoniano_mezcla` che hai già calcolato nella lezione teorica

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import itertools

# Mismo grafo que en la clase teórica: ciclo de seis nodos
grafo = nx.cycle_graph(6)
qubits = [cirq.GridQubit(0, i) for i in range(grafo.number_of_nodes())]

estado_inicial = cirq.Circuit()
for qubit in qubits:
    estado_inicial.append(cirq.H(qubit))

hamiltoniano_mezcla = sum(cirq.PauliString(cirq.X(qubits[i])) for i in range(len(qubits)))

hamiltoniano_coste = grafo.number_of_edges() / 2
for nodo_a, nodo_b in grafo.edges():
    hamiltoniano_coste += cirq.PauliString(0.5 * cirq.Z(qubits[nodo_a]) * cirq.Z(qubits[nodo_b]))

p = 2  # ahora con dos repeticiones del ciclo coste-mezcla

# TODO: declara los cuatro símbolos entrenables con sympy.symbols("gamma1 gamma2 beta1 beta2")

circuito_qaoa = cirq.Circuit()

# TODO: encadena dos llamadas a tfq.util.exponential con circuito_qaoa +=
#       la primera con operators=[hamiltoniano_coste, hamiltoniano_mezcla] y coefficients=[gamma1, beta1]
#       la segunda con coefficients=[gamma2, beta2]

print(circuito_qaoa)

In [ ]:
entradas = tfq.convert_to_tensor([estado_inicial])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_qaoa, hamiltoniano_coste)(capa_entrada)

modelo_qaoa_p2 = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)
modelo_qaoa_p2.compile(loss=tf.keras.losses.MAE, optimizer=tf.keras.optimizers.Adam())

# TODO: entrena el modelo con .fit(entradas, np.array([0]), epochs=200) y guarda el resultado en "historial"

plt.plot(historial.history["loss"])
plt.title("QAOA con p=2 — pérdida durante el entrenamiento")
plt.xlabel("Iteración")
plt.ylabel("Pérdida")
plt.show()

In [ ]:
parametros_entrenados = modelo_qaoa_p2.trainable_variables
circuito_con_datos = tfq.layers.AddCircuit()(entradas, append=circuito_qaoa)

# TODO: muestrea el circuito entrenado con tfq.layers.Sample, pasando
#       symbol_names=[gamma1, gamma2, beta1, beta2], symbol_values=parametros_entrenados y repetitions=1000
#       guarda el resultado en "muestras"

cortes_obtenidos_p2 = []
for bits in muestras.values:
    grupo_uno = [i for i, bit in enumerate(bits) if bit == 1]
    cortes_obtenidos_p2.append(nx.algorithms.cuts.cut_size(grafo, grupo_uno))

corte_maximo = max(
    nx.algorithms.cuts.cut_size(grafo, list(combo))
    for r in range(grafo.number_of_nodes() + 1)
    for combo in itertools.combinations(grafo.nodes(), r)
)

# TODO: imprime el corte medio obtenido con p=2 (np.mean(cortes_obtenidos_p2)) y el corte máximo,
#       y compáralo con el corte medio que obtuviste en la clase teórica con p=1

<details>
<summary>💡 Mostra soluzione</summary>

```python
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import itertools

# Stesso grafo della lezione teorica: ciclo di sei nodi
grafo = nx.cycle_graph(6)
qubits = [cirq.GridQubit(0, i) for i in range(grafo.number_of_nodes())]

estado_inicial = cirq.Circuit()
for qubit in qubits:
    estado_inicial.append(cirq.H(qubit))

hamiltoniano_mezcla = sum(cirq.PauliString(cirq.X(qubits[i])) for i in range(len(qubits)))

hamiltoniano_coste = grafo.number_of_edges() / 2
for nodo_a, nodo_b in grafo.edges():
    hamiltoniano_coste += cirq.PauliString(0.5 * cirq.Z(qubits[nodo_a]) * cirq.Z(qubits[nodo_b]))

p = 2  # ora con due ripetizioni del ciclo costo-mescolamento

# Quattro simboli addestrabili: un gamma e un beta per ogni ripetizione
gamma1, gamma2, beta1, beta2 = sympy.symbols("gamma1 gamma2 beta1 beta2")

circuito_qaoa = cirq.Circuit()
# Prima ripetizione del ciclo costo-mescolamento
circuito_qaoa += tfq.util.exponential(
    operators=[hamiltoniano_coste, hamiltoniano_mezcla],
    coefficients=[gamma1, beta1],
)
# Seconda ripetizione del ciclo costo-mescolamento
circuito_qaoa += tfq.util.exponential(
    operators=[hamiltoniano_coste, hamiltoniano_mezcla],
    coefficients=[gamma2, beta2],
)

print(circuito_qaoa)

entradas = tfq.convert_to_tensor([estado_inicial])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_qaoa, hamiltoniano_coste)(capa_entrada)

modelo_qaoa_p2 = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)
modelo_qaoa_p2.compile(loss=tf.keras.losses.MAE, optimizer=tf.keras.optimizers.Adam())

historial = modelo_qaoa_p2.fit(entradas, np.array([0]), epochs=200)

plt.plot(historial.history["loss"])
plt.title("QAOA con p=2 — perdita durante l'addestramento")
plt.xlabel("Iterazione")
plt.ylabel("Perdita")
plt.show()

parametros_entrenados = modelo_qaoa_p2.trainable_variables
circuito_con_datos = tfq.layers.AddCircuit()(entradas, append=circuito_qaoa)
muestras = tfq.layers.Sample()(
    circuito_con_datos,
    symbol_names=[gamma1, gamma2, beta1, beta2],
    symbol_values=parametros_entrenados,
    repetitions=1000,
)

cortes_obtenidos_p2 = []
for bits in muestras.values:
    grupo_uno = [i for i, bit in enumerate(bits) if bit == 1]
    cortes_obtenidos_p2.append(nx.algorithms.cuts.cut_size(grafo, grupo_uno))

corte_maximo = max(
    nx.algorithms.cuts.cut_size(grafo, list(combo))
    for r in range(grafo.number_of_nodes() + 1)
    for combo in itertools.combinations(grafo.nodes(), r)
)

print("Taglio medio ottenuto con p=2:", np.mean(cortes_obtenidos_p2))
print("Miglior taglio possibile (forza bruta):", corte_maximo)

# Confronta questo risultato con il taglio medio ottenuto nella lezione teorica con p=1:
# con p=2 il circuito ha il doppio di parametri e il doppio di profondità, quindi
# il taglio medio dovrebbe avvicinarsi di più al massimo teorico (anche se il margine
# di miglioramento è piccolo su un grafo semplice come questo ciclo di sei nodi).
```

</details>